# Download Latest Blob Run

This notebook downloads the most recent training run stored under `runs/<timestamp>/` in Azure Blob Storage and saves it locally into `aca_pretraining/model_run/`.

It expects these environment variables to be available in `.env`:

- `AZURE_STORAGE_CONNECTION_STRING`
- `AZURE_BLOB_CONTAINER`


In [2]:
from pathlib import Path

from dotenv import load_dotenv

CWD = Path.cwd().resolve()
SEARCH_ROOTS = [CWD, *CWD.parents]
ENV_CANDIDATES = []

for root in SEARCH_ROOTS:
    ENV_CANDIDATES.append(root / ".env")
    ENV_CANDIDATES.append(root / "aca_pretraining" / ".env")

seen = set()
ENV_CANDIDATES = [path for path in ENV_CANDIDATES if not (str(path) in seen or seen.add(str(path)))]

for env_path in ENV_CANDIDATES:
    if env_path.exists():
        load_dotenv(env_path, override=False)
        NOTEBOOK_DIR = env_path.parent if env_path.parent.name == "aca_pretraining" else CWD
        print(f"Loaded environment from: {env_path}")
        break
else:
    NOTEBOOK_DIR = CWD
    print("No .env file found in the working directory tree; relying on existing environment variables.")

print(f"Working directory: {CWD}")
print(f"Notebook asset directory: {NOTEBOOK_DIR}")

Loaded environment from: C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_pretraining\.env
Working directory: C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch
Notebook asset directory: C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_pretraining


In [3]:
import os
from collections import defaultdict
from pathlib import Path

from azure.storage.blob import BlobServiceClient

CONNECTION_STRING = os.environ["AZURE_STORAGE_CONNECTION_STRING"]
CONTAINER_NAME = os.environ["AZURE_BLOB_CONTAINER"]
RUNS_PREFIX = "runs/"
DOWNLOAD_DIR = NOTEBOOK_DIR / "model_run"
INCLUDE_CHECKPOINTS = False

blob_service = BlobServiceClient.from_connection_string(CONNECTION_STRING)
container_client = blob_service.get_container_client(CONTAINER_NAME)

print(f"Container: {CONTAINER_NAME}")
print(f"Download directory: {DOWNLOAD_DIR}")
print(f"Include checkpoints: {INCLUDE_CHECKPOINTS}")

Container: gpt-scratch
Download directory: C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_pretraining\model_run


In [4]:
blobs = list(container_client.list_blobs(name_starts_with=RUNS_PREFIX))

if not blobs:
    raise RuntimeError("No blobs found under the 'runs/' prefix.")

run_to_blobs = defaultdict(list)

for blob in blobs:
    parts = blob.name.split("/", 2)
    if len(parts) < 2:
        continue
    run_id = parts[1]
    run_to_blobs[run_id].append(blob)

if not run_to_blobs:
    raise RuntimeError("No run folders were detected under the 'runs/' prefix.")

latest_run_id = sorted(run_to_blobs.keys())[-1]
latest_run_blobs = sorted(run_to_blobs[latest_run_id], key=lambda blob: blob.name)
if not INCLUDE_CHECKPOINTS:
    skipped_checkpoint_blobs = [blob for blob in latest_run_blobs if "/checkpoints/" in blob.name]
    latest_run_blobs = [blob for blob in latest_run_blobs if "/checkpoints/" not in blob.name]
else:
    skipped_checkpoint_blobs = []

print(f"Latest run detected: {latest_run_id}")
print(f"Blob count in latest run: {len(latest_run_blobs)}")
print(f"Skipped checkpoint blobs: {len(skipped_checkpoint_blobs)}")

for blob in latest_run_blobs:
    print(blob.name)

Latest run detected: 2026-03-27T06-53-52Z
Blob count in latest run: 6
runs/2026-03-27T06-53-52Z/checkpoints/model_checkpoint_100000.pt
runs/2026-03-27T06-53-52Z/checkpoints/model_checkpoint_150000.pt
runs/2026-03-27T06-53-52Z/checkpoints/model_checkpoint_50000.pt
runs/2026-03-27T06-53-52Z/logs/train.log
runs/2026-03-27T06-53-52Z/metrics/metrics.json
runs/2026-03-27T06-53-52Z/model/pretrain_final.pth


In [5]:
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

downloaded_files = []

for blob in latest_run_blobs:
    relative_path = blob.name.removeprefix(f"runs/{latest_run_id}/")
    if not relative_path:
        continue

    local_path = DOWNLOAD_DIR / relative_path
    local_path.parent.mkdir(parents=True, exist_ok=True)

    with open(local_path, "wb") as file_obj:
        stream = container_client.download_blob(blob.name)
        file_obj.write(stream.readall())

    downloaded_files.append(local_path)
    print(f"Downloaded: {blob.name} -> {local_path}")

print(f"\nDownloaded {len(downloaded_files)} files into {DOWNLOAD_DIR}")

Downloaded: runs/2026-03-27T06-53-52Z/checkpoints/model_checkpoint_100000.pt -> C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_pretraining\model_run\checkpoints\model_checkpoint_100000.pt
Downloaded: runs/2026-03-27T06-53-52Z/checkpoints/model_checkpoint_150000.pt -> C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_pretraining\model_run\checkpoints\model_checkpoint_150000.pt
Downloaded: runs/2026-03-27T06-53-52Z/checkpoints/model_checkpoint_50000.pt -> C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_pretraining\model_run\checkpoints\model_checkpoint_50000.pt
Downloaded: runs/2026-03-27T06-53-52Z/logs/train.log -> C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_pretraining\model_run\logs\train.log
Downloaded: runs/2026-03-27T06-53-52Z/metrics/metrics.json -> C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_pretraining\model_run\me

In [6]:
for path in sorted(DOWNLOAD_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(DOWNLOAD_DIR))

checkpoints\model_checkpoint_100000.pt
checkpoints\model_checkpoint_150000.pt
checkpoints\model_checkpoint_50000.pt
logs\train.log
metrics\metrics.json
model\pretrain_final.pth
